# ***SETUP***

In [1]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Base directory
base_path = "/content/drive/MyDrive/CIC_IOV_2024"

# List of subfolders
folders = [
    "artifacts",
    "lime",
    "shap",
    "models",
    "plots",
    "processed",
    "raw",
    "report"
]

# main folder
os.makedirs(base_path, exist_ok=True)

for folder in folders:
    os.makedirs(os.path.join(base_path, folder), exist_ok=True)

print("Folder structure created successfully!")

Mounted at /content/drive
Folder structure created successfully!


In [2]:
!pip install numpy pandas scikit-learn matplotlib seaborn joblib

# ML Models
!pip install xgboost lightgbm catboost

# Feature Selection (PSO)
!pip install niapy==2.0.0rc17

# Explainability
!pip install shap

# For large datasets
!pip install tqdm

# Complete one-liner:
!pip install numpy pandas scikit-learn matplotlib seaborn joblib xgboost lightgbm catboost niapy==2.0.0rc17 shap tqdm

!pip install lime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.7/200.7 kB 11.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 13.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=7f7bf4a77191af6490ee3f8960c7d6cda7952e9c8556a783a1ee3564697a1ee2
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


# **LIBRARY IMPORTS**

In [4]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from time import time
warnings.filterwarnings('ignore')

# Core ML
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

# Model variety
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import AdaBoostClassifier
import xgboost as xgb
import lightgbm as lgb
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import joblib

# Explainability
import shap
import lime
import lime.lime_tabular


In [7]:
input_files = [
    "/content/drive/MyDrive/CIC_IOV_2024/raw/decimal_DoS.csv",
    "/content/drive/MyDrive/CIC_IOV_2024/raw/decimal_benign.csv",
    "/content/drive/MyDrive/CIC_IOV_2024/raw/decimal_spoofing-GAS.csv",
    "/content/drive/MyDrive/CIC_IOV_2024/raw/decimal_spoofing-RPM.csv",
    "/content/drive/MyDrive/CIC_IOV_2024/raw/decimal_spoofing-SPEED.csv",
    "/content/drive/MyDrive/CIC_IOV_2024/raw/decimal_spoofing-STEERING_WHEEL.csv",
]

In [ ]:
# Output path
output_path = "/content/drive/MyDrive/CIC_IOV_2024/processed/raw_concatenated.csv"

# Read and concatenate
df_list = []
for file in input_files:
    print(f"Reading: {file}")
    df = pd.read_csv(file)
    df_list.append(df)

combined_df = pd.concat(df_list, axis=0, ignore_index=True)

# Save to processed folder
combined_df.to_csv(output_path, index=False)


In [16]:
output_path = "/content/drive/MyDrive/CIC_IOV_2024/processed/raw_concatenated.csv"

# Check if file exists
if os.path.exists(output_path):
    print("File already exists. Loading...")
    combined_df = pd.read_csv(output_path)
else:
    print("File not found. Creating concatenated dataset...")
    df_list = []

    for file in input_files:
        print(f"Reading: {file}")
        df = pd.read_csv(file)
        df_list.append(df)

    combined_df = pd.concat(df_list, axis=0, ignore_index=True)

    combined_df.to_csv(output_path, index=False)
    print("File created and saved successfully!")

print(f"Shape of dataset: {combined_df.shape}")

File already exists. Loading...
Shape of dataset: (1408219, 13)


# ***DATA ANALYSIS***

In [17]:
df1 = pd.read_csv("/content/drive/MyDrive/CIC_IOV_2024/processed/raw_concatenated.csv")

# size
print(df1.shape)

# columns
print(df1.columns)

# types
print(df1.dtypes)

(1408219, 13)
Index(['ID', 'DATA_0', 'DATA_1', 'DATA_2', 'DATA_3', 'DATA_4', 'DATA_5',
       'DATA_6', 'DATA_7', 'label', 'category', 'specific_class',
       'source_file'],
      dtype='object')
ID                 int64
DATA_0             int64
DATA_1             int64
DATA_2             int64
DATA_3             int64
DATA_4             int64
DATA_5             int64
DATA_6             int64
DATA_7             int64
label             object
category          object
specific_class    object
source_file       object
dtype: object


In [23]:
print(df1.isnull().sum())

ID                0
DATA_0            0
DATA_1            0
DATA_2            0
DATA_3            0
DATA_4            0
DATA_5            0
DATA_6            0
DATA_7            0
label             0
category          0
specific_class    0
source_file       0
dtype: int64


In [25]:
#Basic stats

df1.describe()

,ID,DATA_0,DATA_1,DATA_2,DATA_3,DATA_4,DATA_5,DATA_6,DATA_7
count,1.408219e+06,1.408219e+06,1.408219e+06,1.408219e+06,1.408219e+06,1.408219e+06,1.408219e+06,1.408219e+06,1.408219e+06
mean,5.372079e+02,7.108660e+01,6.998925e+01,5.501127e+01,5.745364e+01,4.528517e+01,5.388261e+01,7.174914e+01,6.027477e+01
std,3.224800e+02,8.897717e+01,9.558374e+01,7.276584e+01,9.032077e+01,6.445835e+01,9.433612e+01,1.016872e+02,9.996547e+01
min,6.500000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,3.570000e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,5.160000e+02,1.600000e+01,1.200000e+01,1.300000e+01,0.000000e+00,6.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,5.780000e+02,1.270000e+02,1.280000e+02,1.250000e+02,9.200000e+01,8.600000e+01,6.300000e+01,1.380000e+02,8.000000e+01
max,1.438000e+03,2.550000e+02,2.550000e+02,2.550000e+02,2.550000e+02,2.550000e+02,2.550000e+02,2.550000e+02,2.550000e+02


In [29]:
#Categorical summary

print(df1.describe(include='object'))

          label category specific_class         source_file
count   1408219  1408219        1408219             1408219
unique        2        3              6                   6
top      BENIGN   BENIGN         BENIGN  decimal_benign.csv
freq    1223737  1223737        1223737             1223737


In [52]:
#Simple distribution

save_path = "/content/drive/MyDrive/CIC_IOV_2024/plots/data analysis"
os.makedirs(save_path, exist_ok=True)

df1.hist(figsize=(10,8))
plt.savefig(f"{save_path}/hist.png")
plt.close()

In [32]:
print(df1['label'].value_counts())

label
BENIGN    1223737
ATTACK     184482
Name: count, dtype: int64


In [34]:
#Class Percentage

print(df1['label'].value_counts(normalize=True) * 100)

label
BENIGN    86.899623
ATTACK    13.100377
Name: proportion, dtype: float64


In [55]:
#Class Distribution

df1['label'].value_counts().plot(kind='bar')
plt.savefig(f"{save_path}/class_dist.png")
plt.close()

In [40]:
print(df1.describe().T)

            count        mean         std   min    25%    50%    75%     max
ID      1408219.0  537.207946  322.479994  65.0  357.0  516.0  578.0  1438.0
DATA_0  1408219.0   71.086599   88.977175   0.0    0.0   16.0  127.0   255.0
DATA_1  1408219.0   69.989250   95.583743   0.0    0.0   12.0  128.0   255.0
DATA_2  1408219.0   55.011272   72.765838   0.0    0.0   13.0  125.0   255.0
DATA_3  1408219.0   57.453638   90.320766   0.0    0.0    0.0   92.0   255.0
DATA_4  1408219.0   45.285167   64.458350   0.0    0.0    6.0   86.0   255.0
DATA_5  1408219.0   53.882613   94.336120   0.0    0.0    0.0   63.0   255.0
DATA_6  1408219.0   71.749144  101.687183   0.0    0.0    0.0  138.0   255.0
DATA_7  1408219.0   60.274769   99.965467   0.0    0.0    0.0   80.0   255.0


In [54]:
#Correlation Heatmap
corr = df1.corr(numeric_only=True)

plt.figure(figsize=(10,8))
plt.imshow(corr)

for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        plt.text(j, i, round(corr.iloc[i, j], 2),
                 ha='center', va='center')

plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)

plt.colorbar()
plt.title("Correlation Matrix")

plt.savefig(f"{save_path}/corr.png")
plt.close()

In [46]:
# highly correlated pairs

high = np.where(abs(corr) > 0.8)

for i, j in zip(*high):
    if i < j:
        print(corr.index[i], "-", corr.columns[j], ":", corr.iloc[i, j])

DATA_3 - DATA_5 : 0.8599981148398842


# ***DATA PREPROCESSING***

In [56]:
cols = ['DATA_0','DATA_1','DATA_2','DATA_3','DATA_4','DATA_5','DATA_6','DATA_7','specific_class']
df = pd.read_csv(os.path.join(base_path, "processed/raw_concatenated.csv"))[cols]
df = df.rename(columns={'specific_class': 'Label'})

print(" FULL DATASET:")
print(f"Rows: {len(df):,}")
print("Classes:", df['Label'].value_counts().to_dict())

X_full = df.drop('Label', axis=1).values
y_full = df['Label'].values
feature_names = df.drop('Label', axis=1).columns.tolist()

 FULL DATASET:
Rows: 1,408,219
Classes: {'BENIGN': 1223737, 'DoS': 74663, 'RPM': 54900, 'SPEED': 24951, 'STEERING_WHEEL': 19977, 'GAS': 9991}


In [57]:
# PURE STRATIFIED SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full,
    test_size=0.2,              # 80-20
    stratify=y_full,           # Preserve ALL classes
    random_state=42
)

print(f" Train: {len(X_train):,} rows | Test: {len(X_test):,} rows")
print("Train classes:", pd.Series(y_train).value_counts().sort_index().to_dict())
print("Test classes:", pd.Series(y_test).value_counts().sort_index().to_dict())

# Encode labels
le = LabelEncoder().fit(y_train)
y_train_enc = le.transform(y_train)
y_test_enc = le.transform(y_test)

 Train: 1,126,575 rows | Test: 281,644 rows
Train classes: {'BENIGN': 978989, 'DoS': 59730, 'GAS': 7993, 'RPM': 43920, 'SPEED': 19961, 'STEERING_WHEEL': 15982}
Test classes: {'BENIGN': 244748, 'DoS': 14933, 'GAS': 1998, 'RPM': 10980, 'SPEED': 4990, 'STEERING_WHEEL': 3995}


In [67]:
save_path1 = "/content/drive/MyDrive/CIC_IOV_2024/plots/data preprocessing"
os.makedirs(save_path, exist_ok=True)

In [68]:
# TRAIN VS TEST BAR PLOT

train_counts = pd.Series(y_train).value_counts().sort_index()
test_counts = pd.Series(y_test).value_counts().sort_index()

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
train_counts.plot(kind='bar')
plt.title("Train Classes")

plt.subplot(1,2,2)
test_counts.plot(kind='bar')
plt.title("Test Classes")

plt.tight_layout()

plt.savefig(f"{save_path1}/train_test_bar.png")
plt.close()

In [69]:
# Combined Train vs Test distribution

plt.figure(figsize=(6,4))

pd.Series(y_train).value_counts().plot(kind='bar', alpha=0.6, label='Train')
pd.Series(y_test).value_counts().plot(kind='bar', alpha=0.6, label='Test')

plt.legend()
plt.title("Train vs Test Distribution")

plt.savefig(f"{save_path1}/train_vs_test.png")
plt.close()

In [73]:
# Train pie chart

pd.Series(y_train).value_counts().plot(kind='pie', autopct='%1.1f%%')
plt.title("Train Class Distribution")
plt.ylabel("")

plt.savefig(f"{save_path1}/train_pie.png")
plt.close()

In [71]:
# Test pie chart

pd.Series(y_test).value_counts().plot(kind='pie', autopct='%1.1f%%')
plt.title("Test Class Distribution")
plt.ylabel("")

plt.savefig(f"{save_path1}/test_pie.png")
plt.close()

In [72]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

classes = np.unique(y_train_enc)
weights = compute_class_weight('balanced', classes=classes, y=y_train_enc)
class_weight_dict = dict(zip(classes, weights))

np.savez_compressed(os.path.join(base_path, 'artifacts/full_split.npz'),
                   X_train_scaled=X_train_scaled, X_test_scaled=X_test_scaled,
                   y_train_enc=y_train_enc, y_test_enc=y_test_enc,
                   feature_names=np.array(feature_names), classes=np.array(classes),
                   class_weights=np.array(weights))
print("FULL SPLIT SAVED")

FULL SPLIT SAVED


In [59]:
# Load data
data = np.load(os.path.join(base_path, 'artifacts/full_split.npz'), allow_pickle=True)
X_train_scaled = data['X_train_scaled']
y_train_enc = data['y_train_enc']
feature_names = data['feature_names'].tolist()
n_features = X_train_scaled.shape[1]

print(f" PSO/Firefly on FULL {X_train_scaled.shape[0]:,} rows × {n_features} features")

 PSO/Firefly on FULL 1,126,575 rows × 8 features
